# 00 — First look at the raw data

Dataset: *Credit Card Transactions Fraud Detection* (Kaggle, kartik2112). Simulated card transactions (Sparkov generator), already split into `fraudTrain.csv` and `fraudTest.csv`.

Goal of this notebook: understand **structure, types, target balance and data quality** before designing the Bronze/Silver layers. No cleaning happens here.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/fraudTrain.csv")
df.shape

(1296675, 23)

## 1. Size and memory footprint

In [2]:
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"rows: {len(df):,}   columns: {df.shape[1]}   in-memory: {mem_mb:,.0f} MB")

rows: 1,296,675   columns: 23   in-memory: 427 MB


## 2. Column types

`pd.read_csv` infers types. Things to watch: dates loaded as strings, IDs loaded as numbers.

In [3]:
df.dtypes.to_frame("dtype").assign(n_unique=df.nunique())

,dtype,n_unique
Unnamed: 0,int64,1296675
trans_date_trans_time,str,1274791
cc_num,int64,983
merchant,str,693
category,str,14
amt,float64,52928
first,str,352
last,str,481
gender,str,2
street,str,983


## 3. First rows

In [4]:
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


## 4. Target: fraud ratio (`is_fraud`)

In [5]:
counts = df["is_fraud"].value_counts()
ratio = df["is_fraud"].value_counts(normalize=True)
pd.DataFrame({"count": counts, "share_%": (ratio * 100).round(3)})

,count,share_%
is_fraud,,
0,1289169,99.421
1,7506,0.579


In [6]:
n_fraud = counts[1]
print(f"Fraud rate: {ratio[1]:.4%}  ->  about 1 fraud per {len(df) // n_fraud} transactions")
print(f"A model that always predicts 'not fraud' would score {ratio[0]:.2%} accuracy.")

Fraud rate: 0.5789%  ->  about 1 fraud per 172 transactions
A model that always predicts 'not fraud' would score 99.42% accuracy.


Same check on the test file, to confirm the split has a similar class balance.

In [7]:
df_test = pd.read_csv("../data/raw/fraudTest.csv")
print(f"test rows: {len(df_test):,}")
print(f"test fraud rate: {df_test['is_fraud'].mean():.4%}")
print("same columns as train:", list(df_test.columns) == list(df.columns))

test rows: 555,719
test fraud rate: 0.3860%
same columns as train: True


## 5. Missing values

`isna()` catches real NaNs. Empty / whitespace-only strings are a common hidden form of missing data in CSVs, so check those too.

In [8]:
missing = pd.DataFrame({
    "n_null": df.isna().sum(),
    "n_blank_str": df.select_dtypes("str").apply(lambda s: s.str.strip().eq("").sum()),
}).fillna(0).astype(int)
missing["pct_null"] = (missing["n_null"] / len(df) * 100).round(3)
missing.sort_values("n_null", ascending=False)

,n_null,n_blank_str,pct_null
Unnamed: 0,0,0,0.0
amt,0,0,0.0
category,0,0,0.0
cc_num,0,0,0.0
city,0,0,0.0
city_pop,0,0,0.0
dob,0,0,0.0
first,0,0,0.0
gender,0,0,0.0
is_fraud,0,0,0.0


In [9]:
print("total missing cells (train):", int(df.isna().sum().sum()))
print("total missing cells (test): ", int(df_test.isna().sum().sum()))

total missing cells (train): 0
total missing cells (test):  0


## 6. Quick sanity checks (inputs for the Silver layer design)

In [10]:
ts = pd.to_datetime(df["trans_date_trans_time"])
ts_test = pd.to_datetime(df_test["trans_date_trans_time"])
print("train period:", ts.min(), "->", ts.max())
print("test period: ", ts_test.min(), "->", ts_test.max())
print()
print("'Unnamed: 0' is just a row index:", (df["Unnamed: 0"] == range(len(df))).all())
print("trans_num unique (candidate primary key):", df["trans_num"].is_unique)
print("fully duplicated rows:", int(df.duplicated().sum()))
print("distinct cards:", df["cc_num"].nunique(), "  distinct merchants:", df["merchant"].nunique())

train period: 2019-01-01 00:00:18 -> 2020-06-21 12:13:37
test period:  2020-06-21 12:14:25 -> 2020-12-31 23:59:34

'Unnamed: 0' is just a row index: True


trans_num unique (candidate primary key): True


fully duplicated rows: 0
distinct cards: 983   distinct merchants: 693


In [11]:
# unix_time vs the human-readable timestamp: are they the same instant?
unix_as_dt = pd.to_datetime(df["unix_time"], unit="s")
gap = ts - unix_as_dt
gap.describe()

count                      1296675
mean     2556 days 06:53:31.344091
std         0 days 10:51:30.900633
min             2556 days 00:00:00
25%             2556 days 00:00:00
50%             2556 days 00:00:00
75%             2557 days 00:00:00
max             2557 days 00:00:00
dtype: object

In [12]:
df.groupby("is_fraud")["amt"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,67.67,154.01,1.00,9.61,47.28,82.54,28948.90
1,7506.0,531.32,390.56,1.06,245.66,396.50,900.88,1376.04


## Takeaways

**Structure**
- Train: 1,296,675 rows × 23 columns (~430 MB in pandas). Test: 555,719 rows, same schema.
- The split is **by time**: train = Jan 2019 → 21 Jun 2020, test = 21 Jun 2020 → Dec 2020. No overlap, so it is a realistic "train on the past, predict the future" setup.
- Only 983 cards and 693 merchants → many transactions per card. Good for behavioural features (velocity, deviation from the card's usual spend).

**Target**
- Fraud rate is **0.58%** in train (≈1 in 172) and **0.39%** in test. Heavy class imbalance, and the rate drifts between periods.
- Always predicting "not fraud" gives 99.4% accuracy, so accuracy is useless. Use precision/recall, PR-AUC, and cost-based metrics.
- Fraudulent amounts are much larger (median $397 vs $47) but capped around $1,376, while legit amounts go up to ~$29k. `amt` will be a strong feature, but it is not a simple threshold.

**Data quality**
- **No nulls and no blank strings** anywhere. That is unusual for real data and reflects that the dataset is simulated.
- `trans_num` is unique, so it is the natural primary key. No duplicate rows.
- `Unnamed: 0` is a leftover pandas index. Drop it in Silver.
- `unix_time` is **not** the same instant as `trans_date_trans_time` (shifted by roughly 7 years). Use `trans_date_trans_time` as the source of truth; document the inconsistency.
- Type fixes needed in Silver: `trans_date_trans_time` and `dob` → datetime; `cc_num` and `zip` are identifiers, not numbers → string; `category`, `gender`, `state` → categorical.
- All merchant names start with `fraud_`. This is an artifact of the generator, not a label leak (it appears on legit rows too).

**Privacy**
- The data contains PII-like fields (`first`, `last`, `street`, `dob`, `cc_num`). Plan to hash or drop them in Silver/Gold. This is a good governance point to show in the DE part.
